In [1]:
import pandas as pd
import numpy as np

orcamentos = pd.read_excel("controle_orc.xlsm", skiprows=5)

orcamentos.head()

c:\Users\Fernando\Documents\Projetos\projeto_plataforma_integrada\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,INDEX,MÊS,CLIENTE,OBRA,CONTATO,CELULAR,TERMÔMETRO DE CONFIANÇA DE VENDA,STATUS COMERCIAL,ENTRADA,SOLICITANTE,...,ENVIADO AO COMERCIAL,TECNICO,FORNECEDOR,VALOR ORÇAMENTO (R$),PESO (kg),AREA (m²),REVISAO,PASTA,Status_norm,RELAÇÃO DE CLIENTES
0,1,2024-07-01,M BOKEL CONSTRUTORA,SERRA DA MESA,NaN,NaN,NaN,CANCELADO,2025-01-21,DAILOR,...,2025-02-04,FERNANDO,MERCADO,7112448.36,91977.7,17.00,1.0,Tecnica\01 ORÇAMENTOS\02 CONSTRUTORA\M BOKEL C...,CANCELADO,CONSTRUTORA
1,2,2025-01-01,RECALL ESQUADRIAS,ABSOLUTO,NaN,NaN,NaN,CANCELADO,2025-06-27,ELAINE,...,2025-07-10,ELIAS,CDA,2295223.94,31025.24,3031.90,1.0,Tecnica\01 ORÇAMENTOS\05 SERRALHERIAS\05 RECAL...,CANCELADO,SERRALHERIA
2,3,2025-01-01,APOIO ENGENHARIA,CHÁCARA BERNADETE,NaN,NaN,NaN,CANCELADO,2025-01-08,FRED,...,2025-01-13,FERNANDO,CDA,95481.12,1164.7,146.11,1.0,Tecnica\01 ORÇAMENTOS\01 VAREJO\2025\ALUBRASA\...,CANCELADO,CONSTRUTORA
3,4,2025-01-01,RIDAL,GALPÃO SKS,NaN,NaN,NaN,CANCELADO,2025-01-14,FRED,...,2025-01-28,CRISTIAN,CBA,NaN,7429.26,573.44,0.0,Tecnica\01 ORÇAMENTOS\01 VAREJO\2025\ALUBRASA\...,CANCELADO,NaN
4,5,2025-01-01,STYLOS ENGENHARIA,LIV 635,carlos.issa@stylos.com.br,CARLOS ISSA (61) 99944-6367,NaN,FECHADO,2025-01-15,FRED,...,2025-02-04,CRISTIAN,CBA,1100000,4654.245,2001.57,0.0,Tecnica\01 ORÇAMENTOS\02 CONSTRUTORA\STYLOS EN...,FECHADO,CONSTRUTORA


In [2]:
orcamentos.columns

Index(['INDEX', 'MÊS', 'CLIENTE', 'OBRA', 'CONTATO', 'CELULAR',
       'TERMÔMETRO DE CONFIANÇA DE VENDA', 'STATUS COMERCIAL', 'ENTRADA',
       'SOLICITANTE', 'INÍCIO', 'PRAZO DEPTO. TÉCNICO',
       'STATUS DO LEVANTAMENTO TÉCNICO', 'CODIGO', 'CONCLUSÃO DEPTO. TÉCNICO',
       'ENVIADO AO COMERCIAL', 'TECNICO', 'FORNECEDOR', 'VALOR ORÇAMENTO (R$)',
       'PESO (kg)', 'AREA (m²)', 'REVISAO', 'PASTA', 'Status_norm',
       'RELAÇÃO DE CLIENTES '],
      dtype='object')

In [3]:
import pandas as pd
import re
import unicodedata

# =========
# 1) CARREGAR
# =========
orcamentos = pd.read_excel("controle_orc.xlsm", skiprows=5)

# =========
# 2) FUNÇÕES AUXILIARES
# =========
def strip_accents(s: str) -> str:
    """Remove acentos de uma string."""
    if not isinstance(s, str):
        s = str(s)
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join([c for c in nfkd if not unicodedata.combining(c)])

def normalize_colname(s: str) -> str:
    """
    Normaliza o nome da coluna:
    - remove acentos
    - lowercase
    - troca qualquer caractere não [a-z0-9] por _
    - colapsa múltiplos _ e remove _ das pontas
    """
    s = strip_accents(s).lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

# =========
# 3) NORMALIZAR NOMES BASE
# =========
orig_cols = list(orcamentos.columns)
norm_cols = [normalize_colname(c) for c in orig_cols]
rename_map_base = dict(zip(orig_cols, norm_cols))
orcamentos = orcamentos.rename(columns=rename_map_base)

# =========
# 4) PREFIXOS ESPECIAIS
#    Observação: o pedido fala em "sufixo 'dt_*'/'st_*'",
#    mas aqui aplico como PREFIXO (por ex.: dt_inicio, st_status_comercial),
#    que é o uso mais comum. Se preferir como sufixo, troque a lógica abaixo.
# =========

# Heurística para detectar colunas de data e de status
date_keywords = {"entrada", "inicio", "prazo", "conclusao", "enviado"}  # ajuste livre conforme necessário
status_keyword = "status"

final_cols = {}
for c in orcamentos.columns:
    base = c
    # status?
    if status_keyword in c:
        new_name = f"st_{base}"
    # datas?
    elif any(kw in c for kw in date_keywords):
        new_name = f"dt_{base}"
    else:
        new_name = base
    final_cols[c] = new_name

orcamentos = orcamentos.rename(columns=final_cols)

# =========
# 5) REMOVER COLUNAS SOLICITADAS
#    (após normalização/prefixos)
#    - contato
#    - celular
#    - termometro de confianca de venda
#    - pasta
#    - status_norm
# =========
to_drop_raw = [
    "contato",
    "celular",
    "termometro_de_confianca_de_venda",
    "pasta",
    "status_norm",
]

# Como algumas podem ter recebido prefixo (ex.: st_status_norm), tratamos as duas formas:
to_drop = set()
for col in to_drop_raw:
    to_drop.add(col)            # normalizada simples
    to_drop.add(f"st_{col}")    # caso tenha sido marcada como status

cols_existentes = [c for c in orcamentos.columns if c in to_drop]
orcamentos = orcamentos.drop(columns=cols_existentes, errors="ignore")

# =========
# (Opcional) converter colunas dt_* para datetime
# =========
for c in orcamentos.columns:
    if c.startswith("dt_"):
        # tenta converter silenciosamente
        orcamentos[c] = pd.to_datetime(orcamentos[c], errors="ignore", dayfirst=True)

# Verificação rápida
print(orcamentos.columns.tolist())
orcamentos.head()


['index', 'mes', 'cliente', 'obra', 'st_status_comercial', 'dt_entrada', 'solicitante', 'dt_inicio', 'dt_prazo_depto_tecnico', 'st_status_do_levantamento_tecnico', 'codigo', 'dt_conclusao_depto_tecnico', 'dt_enviado_ao_comercial', 'tecnico', 'fornecedor', 'valor_orcamento_r', 'peso_kg', 'area_m2', 'revisao', 'relacao_de_clientes']


c:\Users\Fernando\Documents\Projetos\projeto_plataforma_integrada\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
C:\Users\Fernando\AppData\Local\Temp\ipykernel_13400\2517972380.py:99: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  orcamentos[c] = pd.to_datetime(orcamentos[c], errors="ignore", dayfirst=True)
C:\Users\Fernando\AppData\Local\Temp\ipykernel_13400\2517972380.py:99: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  orcamentos[c] = pd.to_datetime(orcamentos[c], errors="ignore", dayfirst=True)
C:\Users\Fernando\AppData\Local\Temp\ipykernel_13400\2517972380.py:99: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_dat

,index,mes,cliente,obra,st_status_comercial,dt_entrada,solicitante,dt_inicio,dt_prazo_depto_tecnico,st_status_do_levantamento_tecnico,codigo,dt_conclusao_depto_tecnico,dt_enviado_ao_comercial,tecnico,fornecedor,valor_orcamento_r,peso_kg,area_m2,revisao,relacao_de_clientes
0,1,2024-07-01,M BOKEL CONSTRUTORA,SERRA DA MESA,CANCELADO,2025-01-21,DAILOR,2025-02-04 00:00:00,2025-01-30,CONCLUIDO,ALUB-25-02-0016,2025-02-04 00:00:00,2025-02-04,FERNANDO,MERCADO,7112448.36,91977.7,17.00,1.0,CONSTRUTORA
1,2,2025-01-01,RECALL ESQUADRIAS,ABSOLUTO,CANCELADO,2025-06-27,ELAINE,2025-07-09 00:00:00,2025-01-29,CONCLUIDO,ALUB-25-06-0099,2025-07-10 00:00:00,2025-07-10,ELIAS,CDA,2295223.94,31025.24,3031.90,1.0,SERRALHERIA
2,3,2025-01-01,APOIO ENGENHARIA,CHÁCARA BERNADETE,CANCELADO,2025-01-08,FRED,2025-01-10 00:00:00,2025-01-13,CONCLUIDO,ALUB-25-01-0002_01,2025-01-13 00:00:00,2025-01-13,FERNANDO,CDA,95481.12,1164.7,146.11,1.0,CONSTRUTORA
3,4,2025-01-01,RIDAL,GALPÃO SKS,CANCELADO,2025-01-14,FRED,2025-01-21 00:00:00,2025-01-27,CONCLUIDO,ALUB-25-01-0009,2025-01-27 00:00:00,2025-01-28,CRISTIAN,CBA,NaN,7429.26,573.44,0.0,NaN
4,5,2025-01-01,STYLOS ENGENHARIA,LIV 635,FECHADO,2025-01-15,FRED,2025-01-27 00:00:00,2025-01-30,CONCLUIDO,ALUB-25-01-0014,2025-02-04 00:00:00,2025-02-04,CRISTIAN,CBA,1100000,4654.245,2001.57,0.0,CONSTRUTORA


In [4]:
orcamentos.to_csv("../data/controle_orc_final.xlsx")